# mcp

> MCP server for kosha — repo + package context search and call-graph queries as MCP tools, so Claude, Codex, and any other MCP client can drive the full index.

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

The server wraps the same `Kosha` methods the CLI and daemon expose, over stdio. `mcp` ships with kosha, so no extra is needed. kosha indexes the *current* repo and its venv, so launch it from the project root with the project's environment (`uv run` does both):

```sh
uv add --dev koshas
uv run kosha-mcp            # stdio server; --http for Streamable HTTP
```

In [ ]:
#| export
"""MCP server for kosha — repo + package context search and call-graph tools over stdio.
Run `kosha-mcp` from the target repo's root in its venv (e.g. `uv run kosha-mcp`).

Docs: https://vedicreader.github.io/kosha/mcp.html.md"""

import sys
from kosha.core import Kosha, env_pkg_versions, pkg_url as _pkg_url
from mcp.server.fastmcp import FastMCP

In [ ]:
#| export
mcp = FastMCP('kosha', instructions=(
    'Persistent memory of this repo and its installed packages (FTS5 + vector search + call graph). '
    'Call status first; sync if anything is stale. Before writing new code, env_context checks whether '
    'a dependency already does it; context/repo_context find existing patterns (pagerank = blast radius); '
    'node_info/short_path/api_paths answer call-graph questions; where_to_add gives file:line insertion '
    'points. Queries mix natural language with filters: package:name path:sub lang:py type:FunctionDef; '
    'package!:name hard-filters to one package.'))

_k = None
def _kosha():
    'Lazy Kosha singleton — the embedder cold-start (~3-5s) happens once per server, not per call.'
    global _k
    if _k is None: _k = Kosha()
    return _k

def _jsonable(o):
    'Recursively convert kosha result objects (L, sets, ...) to JSON-serializable types.'
    if isinstance(o, (set, frozenset)): return sorted(o)
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (str, int, float, bool)) or o is None: return o
    if hasattr(o, '__iter__'): return [_jsonable(x) for x in o]
    return str(o)

## Index management

In [ ]:
#| export
@mcp.tool()
def status() -> dict:
    "Index freshness: {files, packages, graph_nodes, stale_files, stale_pkgs}. Call this first; if anything is stale, sync before querying (stale looks like missing)."
    return _jsonable(_kosha().status())

@mcp.tool()
def sync(dir:str|None=None, pkgs:list|None=None, embed:bool=True, force:bool=False, sync_graph:bool=False) -> dict:
    "Sync repo + env packages + call graph into the .kosha/ index. Incremental — only changed files and new package versions re-index. Returns post-sync status."
    k = _kosha()
    k.sync(dir=dir, pkgs=pkgs, embed=embed, force=force, sync_graph=sync_graph)
    return _jsonable(k.status())

## Context search

In [ ]:
#| export
@mcp.tool()
def context(query:str, limit:int=10, repo:bool=True, env:bool=True, graph:bool=True, compact:bool=False) -> list:
    "Fan-out semantic + keyword search over repo and installed packages, graph-enriched (callers/callees/pagerank). The right default once a task touches more than one module. compact=True returns slim dicts (mod_name, signature, docstring, lineno) for triage."
    return _jsonable(_kosha().context(query, limit=limit, repo=repo, env=env, graph=graph, compact=compact))

@mcp.tool()
def repo_context(query:str, limit:int=10) -> list:
    "Semantic + keyword search over indexed repo code only."
    return _jsonable(_kosha().repo_context(query, limit=limit))

@mcp.tool()
def env_context(query:str, limit:int=10) -> list:
    "Semantic search over installed packages only — check whether a dependency already does something before implementing it. Bare package names in the query auto-detect as soft filters."
    return _jsonable(_kosha().env_context(query, limit=limit))

@mcp.tool()
def where_to_add(description:str, limit:int=5) -> list:
    "Likely file:line insertion points for new code matching a description, with co_dispatched peers to pattern-match."
    return _jsonable(_kosha().where_to_add(description, limit=limit))

## Call graph

In [ ]:
#| export
@mcp.tool()
def node_info(mod_name:str) -> dict:
    "Callers, callees, co_dispatched peers, and pagerank for a fully-qualified node, e.g. fastcore.basics.merge. pagerank = blast radius: high means load-bearing, touch carefully."
    return _jsonable(_kosha().ni(mod_name))

@mcp.tool()
def neighbors(node:str, depth:int=1) -> list:
    "Callers + callees of a fully-qualified node up to depth hops."
    return _jsonable(_kosha().neighbors(node, depth=depth))

@mcp.tool()
def short_path(src:str, tgt:str) -> list:
    "Shortest call-graph path between two fully-qualified nodes (how do these connect?)."
    return _jsonable(_kosha().short_path(src, tgt))

@mcp.tool()
def public_api(pkg:str, limit:int=200) -> list:
    "A package's real public API — __all__ entries plus @patch-derived methods — with docstrings. Works for a package ('fastcore') or submodule ('fastcore.basics')."
    return _jsonable(_kosha().public_api(pkg, limit=limit))

@mcp.tool()
def top_nodes(pkg:str, k:int=5) -> list:
    "Top-k public API nodes for a package ranked by PageRank in the call graph."
    return _jsonable(_kosha().top_nodes(pkg, k=k))

@mcp.tool()
def api_paths(from_pkg:str, to_pkg:str, k:int=15) -> dict:
    "Shortest call-graph paths from one package's public API to another's — how two packages actually connect."
    return _jsonable(_kosha().api_call_paths(from_pkg, to_pkg, k=k))

@mcp.tool()
def dep_stack(seeds:list|None=None, depth:int=1) -> list:
    "BFS dependency layers from seed packages (default: pyproject deps), ordered by coupling strength."
    return _jsonable(_kosha().dep_stack(seeds=seeds or list(env_pkg_versions().keys()), depth=depth))

@mcp.tool()
def pkg_url(pkg:str) -> str:
    "Best repo/docs URL for an installed package — feed it to a web-fetch tool for docs, changelogs, or migration guides."
    return _pkg_url(pkg) or "not found"

In [ ]:
#| export
def main():
    "Entry point for the `kosha-mcp` console script. stdio by default; pass --http for Streamable HTTP."
    mcp.run(transport='streamable-http' if '--http' in sys.argv[1:] else 'stdio')

## Connect a client

The server must start at the project root with the project's venv, so the launch command is `uv run kosha-mcp` rather than `uvx`.

**Claude Code** (run inside the project)

```sh
claude mcp add kosha -- uv run kosha-mcp
```

**Codex** (`~/.codex/config.toml`; Codex launches servers from your session's working directory, so start it at the project root)

```toml
[mcp_servers.kosha]
command = "uv"
args = ["run", "kosha-mcp"]
```

**Claude Desktop** (`claude_desktop_config.json` — pin the project explicitly)

```json
{"mcpServers": {"kosha": {"command": "uv", "args": ["run", "--project", "/path/to/your/repo", "kosha-mcp"]}}}
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()